## SUBTASK 1 (PO2-453): 
    Implementar função que inspeciona metadados e propriedades do raster, além de converter fornato e aplicar conversão


In [3]:
#PO2-465: 

import rasterio
from rasterio.plot import show
import os
import zipfile
import json

def des_zip(): 
    pasta_destino = r'Orthos'
    arquivo_zip = r'/home/maira/anaconda3/envs/env_onboarding/notebooks/Orthos.zip'
    with zipfile.ZipFile(arquivo_zip, 'r') as zip_ref:
        zip_ref.extractall(pasta_destino)

def inspect_raster(path): 
    with rasterio.open(path) as src:
        return {
            "raster": str(path), 
            "crs": str(src.crs), 
            "bounds": src.bounds,
            "resolution": {
                    "x": src.res[0],
                    "y": src.res[1],
            },
            "nodata": src.nodata,
            "dtype": src.dtypes,
            "unidade": src.crs.linear_units,
        }


all_meta = [] 
root = r'/home/maira/workspace/onboarding/Orthos'
names = os.listdir(root)
for name in names: 
    meta = inspect_raster(os.path.join(root, name))
    all_meta.append(meta)
with open(f'all_orthomosaico.json', 'w', encoding='utf-8') as f:
    json.dump(all_meta, f, indent=4)

## SUBTASK 2 (PO2-464): 
    Implementar função que converte raster e aplica conversão 

In [ ]:
#PO2-464: 

from osgeo import gdal
import os

def convert_raster(input_path, output_path, compression='LZW'):
    """
    Converte e otimiza um raster usando gdal_translate.
    
    Args:
        input_path (str): Caminho do arquivo raster de entrada.
        output_path (str): Caminho para salvar o raster convertido.
        compression (str): Tipo de compressão ('LZW', 'DEFLATE', 'PACKBITS', 'ZSTD').
    """
    
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Arquivo de entrada não encontrado: {input_path}")

    ds = gdal.Open(input_path)
    if ds is None:
        raise Exception(f"Não foi possível abrir o arquivo: {input_path}")

    creation_options = [
        f'COMPRESS={compression}',
        'TILED=YES',
        'BIGTIFF=IF_NEEDED',
        'NUM_THREADS=ALL_CPUS'
    ]
    
    if compression in ['LZW', 'DEFLATE', 'PACKBITS', 'ZSTD']:
        creation_options.append('PREDICTOR=2')

    print(f"Convertendo {input_path} para {output_path} com compressão {compression}...")

    gdal.Translate(
        output_path,
        ds,
        creationOptions=creation_options
    )

    ds = None
    print("Conversão concluída com sucesso.")

input_path = r'/home/maira/anaconda3/envs/env_onboarding/notebooks/Orthos/Orthomosaico.tif'
output_path = r'/home/maira/anaconda3/envs/env_onboarding/notebooks/output/Orthomosaico.tif'
compression = 'LZW'
convert_raster(input_path, output_path, compression ='LZW')